# 00 Core Algorithms and Probabilistic Utilities
This Notebook contains shared functions for reading discrete Bayesian networks, FGCR construction, exact inference, sampling, BDeu estimation, and error bounds. It only defines functions and data structures and does not launch the full experiments.


In [1]:
"""Small, auditable discrete BN implementation. Only NumPy is required.
Conventions: factor axes follow scope; CPT axes are parents then child.
Variable elimination is exact apart from floating-point arithmetic.
"""
from __future__ import annotations
from dataclasses import dataclass
import gzip
import itertools
import math
import re
from pathlib import Path
import numpy as np


@dataclass
class Factor:
    scope: tuple[int, ...]
    values: np.ndarray


@dataclass
class Network:
    names: list[str]
    states: list[list[str]]
    parents: list[tuple[int, ...]]
    cpts: list[np.ndarray]
    normalization_max_deviation: float = 0.

    @property
    def cards(self):
        return [len(s) for s in self.states]

    @property
    def edges(self):
        return {(u, v) for v, pa in enumerate(self.parents) for u in pa}

    def factors(self):
        return [Factor(self.parents[v] + (v,), self.cpts[v])
                for v in range(len(self.names))]


def read_bif(path: Path) -> Network:
    """Parser for the explicit discrete row/table BIF subset used by bnlearn.

    Rejects missing/duplicate rows, wrong cardinalities, and substantial
    normalization errors instead of silently inventing parameters.
    """
    raw = gzip.decompress(path.read_bytes()) if path.suffix == ".gz" else path.read_bytes()
    text = re.sub(r"//[^\n]*", "", raw.decode("utf-8"))
    variables = re.findall(
        r"variable\s+([^\s{]+)\s*\{\s*type\s+discrete\s*\[\s*(\d+)\s*\]\s*\{([^}]+)\}\s*;\s*\}",
        text, flags=re.S)
    if not variables:
        raise ValueError("No discrete BIF variables")
    names = [v[0] for v in variables]
    states = [[x.strip().strip('"') for x in v[2].split(",")] for v in variables]
    assert all(len(st) == int(v[1]) for st, v in zip(states, variables))
    lookup = {n: i for i, n in enumerate(names)}
    parents = [None] * len(names)
    cpts = [None] * len(names)
    max_deviation = 0.
    for header, body in re.findall(r"probability\s*\(([^)]+)\)\s*\{([^}]+)\}", text, re.S):
        h = header.split("|")
        v = lookup[h[0].strip()]
        pa = tuple(lookup[x.strip()] for x in h[1].split(",")) if len(h) == 2 else ()
        shape = tuple(len(states[u]) for u in pa) + (len(states[v]),)
        values = np.full(shape, np.nan)
        if not pa:
            m = re.search(r"table\s+([^;]+);", body)
            if m is None:
                raise ValueError("Missing root table")
            values[:] = [float(x) for x in m[1].split(",")]
        else:
            for config, row in re.findall(r"\(([^)]+)\)\s*([^;]+);", body):
                cells = [x.strip().strip('"') for x in config.split(",")]
                idx = tuple(states[u].index(x) for u, x in zip(pa, cells))
                if len(cells) != len(pa) or np.isfinite(values[idx]).any():
                    raise ValueError("Invalid or duplicated BIF row")
                values[idx] = [float(x) for x in row.split(",")]
        if np.isnan(values).any() or (values < 0).any():
            raise ValueError(f"Missing/negative CPT entry for {names[v]}")
        sums = values.sum(axis=-1, keepdims=True)
        err = float(np.max(np.abs(sums - 1)))
        if err > 1e-6:
            raise ValueError(f"Invalid CPT normalization: {err}")
        max_deviation = max(max_deviation, err)
        parents[v] = pa
        cpts[v] = values / sums  # Record harmless decimal-rounding correction.
    if any(x is None for x in parents):
        raise ValueError("Missing CPT")
    bn = Network(names, states, parents, cpts, max_deviation)
    topological_order(len(names), bn.edges)
    return bn


def topological_order(p, edges, rng=None):
    pa = [set() for _ in range(p)]
    children = [set() for _ in range(p)]
    for u, v in edges:
        pa[v].add(u); children[u].add(v)
    ready = sorted(v for v in range(p) if not pa[v])
    out = []
    while ready:
        j = 0 if rng is None else int(rng.integers(len(ready)))
        v = ready.pop(j); out.append(v)
        for w in sorted(children[v]):
            pa[w].remove(v)
            if not pa[w]:
                ready.append(w)
        ready.sort()
    if len(out) != p:
        raise ValueError("Cyclic graph")
    return out


def fgcr(p, edges, C, beta):
    """Partition-preserving cover/reverse constructor, not an ROS reimplementation."""
    E = set(edges)
    order = list(beta)
    working_pos = {v: i for i, v in enumerate(order)}
    alpha = [v for v in beta if v in C] + [v for v in beta if v not in C]
    pos = {v: i for i, v in enumerate(alpha)}
    parents = [set() for _ in range(p)]
    for u, v in E:
        parents[v].add(u)
    swaps = reversals = additions = 0
    max_degree = max(map(len, parents), default=0)
    for y in reversed(alpha):
        j = working_pos[y]
        while j + 1 < p and pos[order[j + 1]] < pos[y]:
            z = order[j + 1]
            assert y not in C and z in C and (z, y) not in E
            if (y, z) in E:
                py, pz = parents[y].copy(), parents[z] - {y}
                for x in py - pz:
                    E.add((x, z)); parents[z].add(x); additions += 1
                for x in pz - py:
                    E.add((x, y)); parents[y].add(x); additions += 1
                max_degree = max(max_degree, len(parents[y]), len(parents[z]))
                E.remove((y, z)); E.add((z, y))
                parents[z].remove(y); parents[y].add(z)
                reversals += 1
            order[j:j + 2] = [z, y]
            working_pos[z], working_pos[y] = j, j + 1
            swaps += 1; j += 1
            max_degree = max(max_degree, len(parents[y]), len(parents[z]))
    assert order == alpha
    H = {(u, v) for u, v in E if u not in C and v not in C}
    return H, E, dict(swaps=swaps, reversals=reversals, additions=additions,
                      intermediate_max_degree=max_degree)


def separated(p, E, x, y, S):
    """Independent d-separation via ancestral moralization."""
    S = set(S)
    if x in S or y in S:
        raise ValueError("Endpoints in conditioning set")
    anc = {x, y} | S
    while True:
        nxt = anc | {u for u, v in E if v in anc}
        if nxt == anc:
            break
        anc = nxt
    adj = {u: set() for u in anc}
    for u, v in E:
        if u in anc and v in anc:
            adj[u].add(v); adj[v].add(u)
    for v in anc:
        pa = [u for u, w in E if w == v and u in anc]
        for u, w in itertools.combinations(pa, 2):
            adj[u].add(w); adj[w].add(u)
    reached = {x}
    stack = [x]
    while stack:
        for v in adj[stack.pop()] - reached - S:
            reached.add(v); stack.append(v)
    return y not in reached


def ordered_oracle(p, E, C, order):
    out = set()
    for j, v in enumerate(order):
        pred = set(order[:j])
        for u in pred:
            if not separated(p, E, u, v, pred - {u} | set(C)):
                out.add((u, v))
    return out


def multiply(factors, cards):
    if not factors:
        return Factor((), np.array(1.))
    scope = tuple(sorted({v for f in factors for v in f.scope}))
    shape = tuple(cards[v] for v in scope)
    if math.prod(shape) > 20_000_000:
        raise MemoryError(f"Exact factor exceeds declared 20M-cell limit: {shape}")
    values = np.ones(shape)
    for f in factors:
        seq = tuple(v for v in scope if v in f.scope)
        arr = f.values.transpose(tuple(f.scope.index(v) for v in seq)) if seq else f.values
        arr = arr.reshape(tuple(cards[v] if v in f.scope else 1 for v in scope))
        values *= arr
    return Factor(scope, values)


def marginal(factors, query, cards, evidence=None, normalize=True):
    """Greedy minimum-result-size elimination. Deterministic tie-breaking."""
    evidence = evidence or {}
    fs = []
    for f in factors:
        idx = tuple(evidence.get(v, slice(None)) for v in f.scope)
        fs.append(Factor(tuple(v for v in f.scope if v not in evidence), f.values[idx]))
    all_vars = {v for f in fs for v in f.scope}
    eliminate = all_vars - set(query)
    while eliminate:
        def cost(v):
            union = {w for f in fs if v in f.scope for w in f.scope}
            return (math.prod(cards[w] for w in union), len(union), v)
        v = min(eliminate, key=cost)
        hit = [f for f in fs if v in f.scope]
        rest = [f for f in fs if v not in f.scope]
        product = multiply(hit, cards)
        arr = product.values.sum(axis=product.scope.index(v))
        rest.append(Factor(tuple(w for w in product.scope if w != v), arr))
        fs = rest; eliminate.remove(v)
    answer = multiply(fs, cards)
    query = tuple(query)
    if query:
        arr = answer.values.transpose(tuple(answer.scope.index(v) for v in query))
    else:
        arr = answer.values
    if normalize:
        mass = arr.sum()
        if mass <= 0:
            raise ValueError("Impossible evidence")
        arr = arr / mass
    return arr


def sample_joint(bn, n, rng):
    p = len(bn.names)
    X = np.zeros((n, p), dtype=np.int16)
    for v in topological_order(p, bn.edges):
        pa = bn.parents[v]
        rows = bn.cpts[v][tuple(X[:, u] for u in pa)] if pa else np.broadcast_to(bn.cpts[v], (n, bn.cards[v]))
        u = rng.random(n)
        X[:, v] = (u[:, None] > np.cumsum(rows, axis=1)).sum(axis=1)
    return X


def sample_conditional(bn, n, evidence, rng, probability):
    """Exact rejection sampling; reports proposal count, not merely retained n."""
    batches, retained, proposed = [], 0, 0
    while retained < n:
        need = n - retained
        m = min(250_000, max(256, int(math.ceil(1.1 * need / probability))))
        X = sample_joint(bn, m, rng)
        mask = np.ones(m, dtype=bool)
        for v, state in evidence.items():
            mask &= X[:, v] == state
        accepted = X[mask]
        batches.append(accepted[:need])
        retained += min(len(accepted), need)
        proposed += m
        if proposed > 100_000_000:
            raise RuntimeError("Rejection sampling proposal budget exceeded")
    return np.concatenate(batches), proposed


def graph_parents(nodes, E):
    return {v: tuple(sorted(u for u, w in E if w == v)) for v in sorted(nodes)}


def cell_indices(X, pa, cards):
    if not pa:
        return np.zeros(len(X), dtype=np.int64)
    return np.ravel_multi_index(tuple(X[:, v] for v in pa), tuple(cards[v] for v in pa))


def fit_tables(X, parents, cards, iss=10.):
    fit = {}
    for v, pa in parents.items():
        q, r = math.prod(cards[u] for u in pa), cards[v]
        idx = cell_indices(X, pa, cards)
        counts = np.bincount(idx * r + X[:, v], minlength=q * r).reshape(q, r)
        N = counts.sum(axis=1)
        A = iss / q
        phat = (counts + A / r) / (N[:, None] + A)
        fit[v] = dict(pa=pa, q=q, r=r, N=N, phat=phat, A=A)
    return fit


def fitted_factors(fit, cards):
    return [Factor(t["pa"] + (v,), t["phat"].reshape(tuple(cards[u] for u in t["pa"]) + (cards[v],)))
            for v, t in fit.items()]


def logloss(fit, X, cards):
    loss = np.zeros(len(X))
    for v, t in fit.items():
        idx = cell_indices(X, t["pa"], cards)
        loss -= np.log(t["phat"][idx, X[:, v]])
    return float(loss.mean())


def robust_simplex_upper(weights, errors, radius):
    """Maximize w dot errors over simplex intersected with coordinate intervals."""
    lower = np.maximum(0., weights - radius)
    upper = np.minimum(1., weights + radius)
    w = lower.copy()
    remain = max(0., 1 - float(w.sum()))
    for j in np.argsort(-errors, kind="stable"):
        add = min(remain, float(upper[j] - w[j]))
        w[j] += add; remain -= add
        if remain < 1e-14:
            break
    if remain > 1e-8:
        raise AssertionError("Empty confidence simplex")
    return float(w @ errors)


def weighted_bounds(fit, true, n, delta=.05):
    """Oracle and fully empirical simultaneous bounds at confidence 1-delta."""
    Q = sum(t["q"] for t in fit.values())
    L = sum(t["q"] * t["r"] for t in fit.values())
    ell = math.log(4 * L / delta)  # half failure budget for local CPT errors
    radius = math.sqrt(math.log(4 * Q / delta) / (2 * n))
    oracle = empirical = 0.
    pi_min, theta_min = 1., 1.
    rsum = 0
    Amax = 0.
    for v, t in fit.items():
        N, A, r, phat = t["N"], t["A"], t["r"], t["phat"]
        b = np.ones(len(N))
        use = N > 0
        b[use] = np.minimum(1., N[use] / (N[use] + A) * np.sqrt(ell / (2 * N[use]))
                            + A * (r - 1) / ((N[use] + A) * r))
        B = np.minimum(b*b * (1/phat).sum(axis=1), -np.log(phat.min(axis=1)))
        pi, theta = true[v]["pi"], true[v]["theta"]
        oracle += float(pi @ B)
        empirical += robust_simplex_upper(N / n, B, radius)
        supported = pi > 0
        pi_min = min(pi_min, float(pi[supported].min()))
        theta_min = min(theta_min, float(theta[supported].min()))
        rsum += r; Amax = max(Amax, A)
    eps = math.sqrt(math.log(4 * L / delta) / (n * pi_min)) + 2*Amax/(n*pi_min)
    valid = theta_min > 0 and n >= 8/pi_min*math.log(2*Q/delta) and eps <= theta_min/2
    uniform = 2*rsum*eps*eps/theta_min if valid else None
    return dict(oracle_bound=oracle, empirical_bound=empirical, uniform_bound=uniform,
                uniform_applicable=valid, pi_min=pi_min, theta_min=theta_min)


def true_family(bn, evidence, v, pa):
    joint = marginal(bn.factors(), pa + (v,), bn.cards, evidence).reshape(-1, bn.cards[v])
    pi = joint.sum(axis=1)
    theta = np.divide(joint, pi[:, None], out=np.full_like(joint, 1/bn.cards[v]), where=pi[:, None] > 0)
    term = np.zeros_like(theta)
    np.log(theta, out=term, where=theta > 0)
    entropy = -float((joint * term).sum())
    return dict(joint=joint, pi=pi, theta=theta, entropy=entropy)


def cross_entropy(fit, true):
    return sum(-float((true[v]["joint"] * np.log(t["phat"])).sum()) for v, t in fit.items())


def pilot_score(fit, n, delta=.05):
    """Prespecified quantile score; heuristic, not a confidence bound."""
    L = sum(t["q"]*t["r"] for t in fit.values())
    weights = np.concatenate([t["N"][t["N"] > 0] / n for t in fit.values()])
    probs = np.concatenate([t["phat"].ravel() for t in fit.values()])
    pi = float(np.quantile(weights, .1))
    theta = float(np.quantile(probs, .1))
    return math.sqrt(sum(t["r"] for t in fit.values()) * math.log(4*L/delta)/(n*pi*theta))
